# Traductor RNN - Seq2Seq LSTM
## Universidad Autónoma del Caribe (UAC)
### Metodología CRISP-ML(Q)

# FASE 1: Business & Data Understanding
## 1.1 Definición del Problema
- **Objetivo**: Traductor automático Inglés ↔ Español
- **Arquitectura**: Seq2Seq LSTM
- **Métricas**: BLEU Score ≥ 0.30
- **Latencia**: < 2 segundos

In [ ]:
# ========================================
# CELDA 1: Instalación de dependencias
# ========================================

!pip install torch numpy gradio fastapi uvicorn onnxruntime

In [ ]:
# ========================================
# CELDA 2: Importar librerías
# ========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import json
from collections import Counter
import matplotlib.pyplot as plt
import gradio as gr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# FASE 2: Data Preparation
## 2.1 Definir corpus y vocabularios

In [ ]:
# ========================================
# CELDA 3: Corpus y Vocabulario (definiciones)
# ========================================

CORPUS = [
    ("hello", "hola"), ("goodbye", "adios"), ("good morning", "buenos dias"),
    ("good night", "buenas noches"), ("thank you", "gracias"), ("please", "por favor"),
    ("yes", "si"), ("no", "no"), ("i am a student", "soy estudiante"),
    ("where is the library", "donde esta la biblioteca"), ("the exam is difficult", "el examen es dificil"),
    ("i need to study", "necesito estudiar"), ("how are you", "como estas"),
    ("i study at the university", "estudio en la universidad"),
    ("good evening", "buenas tardes"), ("see you later", "hasta luego"),
    ("thank you very much", "muchas gracias"), ("you are welcome", "de nada"),
    ("excuse me", "disculpe"), ("sorry", "lo siento"),
    ("maybe", "quizas"), ("of course", "por supuesto"),
    ("i", "yo"), ("you", "tu"), ("he", "el"), ("she", "ella"),
    ("we", "nosotros"), ("they", "ellos"),
    ("father", "padre"), ("mother", "madre"), ("brother", "hermano"), ("sister", "hermana"),
    ("university", "universidad"), ("class", "clase"), ("professor", "profesor"),
    ("student", "estudiante"), ("exam", "examen"), ("homework", "tarea"),
    ("book", "libro"), ("computer", "computadora"),
    ("one", "uno"), ("two", "dos"), ("three", "tres"), ("four", "cuatro"),
    ("five", "cinco"), ("six", "seis"), ("seven", "siete"), ("eight", "ocho"),
    ("nine", "nueve"), ("ten", "diez"),
    ("monday", "lunes"), ("tuesday", "martes"), ("wednesday", "miercoles"),
    ("thursday", "jueves"), ("friday", "viernes"), ("saturday", "sabado"),
    ("sunday", "domingo"), ("today", "hoy"), ("tomorrow", "manana"),
    ("good", "bueno"), ("bad", "malo"), ("big", "grande"), ("small", "pequeno"),
    ("new", "nuevo"), ("old", "viejo"), ("fast", "rapido"), ("slow", "lento"),
    ("easy", "facil"), ("difficult", "dificil"),
]

for es, en in list(CORPUS):
    if (en, es) not in CORPUS:
        CORPUS.append((en, es))

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

class Vocab:
    def __init__(self):
        self.w2i = {PAD: 0, UNK: 1, SOS: 2, EOS: 3}
        self.i2w = {0: PAD, 1: UNK, 2: SOS, 3: EOS}
        self.n = 4
    def add(self, text):
        for w in text.lower().split():
            if w not in self.w2i:
                self.w2i[w] = self.n
                self.i2w[self.n] = w
                self.n += 1
    def encode(self, text, max_len, sos=False, eos=False):
        ids = []
        if sos: ids.append(self.w2i[SOS])
        for w in text.lower().split():
            ids.append(self.w2i.get(w, self.w2i[UNK]))
        if eos: ids.append(self.w2i[EOS])
        while len(ids) < max_len: ids.append(self.w2i[PAD])
        return ids[:max_len]
    def decode(self, ids):
        ws = []
        for i in ids:
            if torch.is_tensor(i): i = i.item()
            w = self.i2w.get(i, UNK)
            if w not in [PAD, SOS, EOS]: ws.append(w)
        return " ".join(ws)

src_v, tgt_v = Vocab(), Vocab()
for s, t in CORPUS:
    src_v.add(s)
    tgt_v.add(t)

MAX_LEN = 20
EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT = 256, 512, 2, 0.3

print(f"Corpus: {len(CORPUS)} parejas")
print(f"Vocab src: {src_v.n}, tgt: {tgt_v.n}")

In [ ]:
# ========================================
# CELDA 4: Definir modelo (arquitectura)
# ========================================

class Encoder(nn.Module):
    def __init__(self, vs, em, hd, ly, dp):
        super().__init__()
        self.emb = nn.Embedding(vs, em, padding_idx=0)
        self.lstm = nn.LSTM(em, hd, ly, batch_first=True, dropout=dp)
        self.dp = nn.Dropout(dp)
    def forward(self, x):
        e = self.dp(self.emb(x))
        o, (h, c) = self.lstm(e)
        return o, h, c

class Decoder(nn.Module):
    def __init__(self, vs, em, hd, ly, dp):
        super().__init__()
        self.emb = nn.Embedding(vs, em, padding_idx=0)
        self.lstm = nn.LSTM(em, hd, ly, batch_first=True, dropout=dp)
        self.fc = nn.Linear(hd, vs)
        self.dp = nn.Dropout(dp)
    def forward(self, x, h, c):
        e = self.dp(self.emb(x))
        o, (h, c) = self.lstm(e, (h, c))
        return self.fc(o.squeeze(1)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec
    def forward(self, src, tgt, tf=0.5):
        bs = src.shape[0]
        max_len = tgt.shape[1]
        out = torch.zeros(bs, max_len, self.dec.fc.out_features).to(src.device)
        _, h, c = self.enc(src)
        dec_in = tgt[:, 0]
        for t in range(1, max_len):
            o, h, c = self.dec(dec_in.unsqueeze(1), h, c)
            out[:, t] = o
            top1 = o.argmax(1)
            dec_in = tgt[:, t] if np.random.random() < tf else top1
        return out

def create_model():
    enc = Encoder(src_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
    dec = Decoder(tgt_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
    model = Seq2Seq(enc, dec).to(device)
    return model, enc, dec

print("Arquitectura definida")

## 2.2 Verificar/Cargar modelo pre-entrenado

In [ ]:
# ========================================
# CELDA 5: Cargar modelo O entrenar desde cero
# ========================================

MODEL_PATH = "translator_model.pt"
USE_PRETRAINED = os.path.exists(MODEL_PATH)

if USE_PRETRAINED:
    print("✅ Cargando modelo pre-entrenado...")
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    
    # Cargar vocabularios
    src_v.w2i = checkpoint.get('src_vocab', src_v.w2i)
    tgt_v.w2i = checkpoint.get('tgt_vocab', tgt_v.w2i)
    src_v.i2w = checkpoint.get('src_idx2word', src_v.i2w)
    tgt_v.i2w = checkpoint.get('tgt_idx2word', tgt_v.i2w)
    
    # Recrear modelo
    model, enc, dec = create_model()
    model.load_state_dict(checkpoint['model_state_dict'])
    
    avg_bleu = checkpoint.get('bleu_score', 0.0)
    print(f"✅ Modelo cargado! BLEU: {avg_bleu:.2f}")
else:
    print("⚠️ Modelo no encontrado. Debe entrenar.")
    print("   Ejecute las siguientes celdas para entrenar.")
    model, enc, dec = None, None, None

# FASE 3 & 4: Training (solo si no hay modelo)

In [ ]:
# ========================================
# CELDA 6: Entrenar modelo (si es necesario)
# ========================================

if model is None:
    print("🔄 Entrenando modelo... (esto toma ~5-10 minutos)")
    
    model, enc, dec = create_model()
    
    class DS(Dataset):
        def __init__(self, data, sv, tv, ml):
            self.d = [(sv.enc(s, ml), tv.enc(t, ml, True, True)) for s, t in data]
        def __len__(self): return len(self.d)
        def __getitem__(self, i):
            return torch.tensor(self.d[i][0]), torch.tensor(self.d[i][1])
    
    ds = DS(CORPUS, src_v, tgt_v, MAX_LEN)
    dl = DataLoader(ds, batch_size=16, shuffle=True)
    
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    EPOCHS = 50
    losses = []
    
    model.train()
    for ep in range(1, EPOCHS + 1):
        ep_loss = 0
        for src, tgt in dl:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()
            out = model(src, tgt)
            loss = criterion(out.view(-1, out.shape[-1]), tgt.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(dl))
        if ep % 10 == 0:
            print(f"Epoch {ep}/{EPOCHS} - Loss: {losses[-1]:.4f}")
    
    print("✅ Entrenamiento completado!")
else:
    print("✅ Modelo ya cargado, saltando entrenamiento.")

# FASE 5: Evaluation

In [ ]:
# ========================================
# CELDA 7: Evaluación BLEU
# ========================================

def calculate_bleu(ref, hyp):
    rw, hw = ref.lower().split(), hyp.lower().split()
    if not hw: return 0.0
    m = sum(1 for w in hw if w in rw)
    p = m / len(hw)
    bp = min(1.0, np.exp(1 - len(rw) / max(len(hw), 1))
    return bp * p

model.eval()
test_samples = [
    ("hello", "hola"), ("thank you", "gracias"),
    ("i am a student", "soy estudiante"),
    ("where is the library", "donde esta la biblioteca"),
]

total_bleu = 0
with torch.no_grad():
    for src_text, tgt_text in test_samples:
        enc_in = torch.tensor([src_v.encode(src_text, MAX_LEN)]).to(device)
        _, h, c = enc(enc_in)
        dec_in = torch.tensor([tgt_v.w2i[SOS]]).to(device)
        result = []
        for _ in range(MAX_LEN):
            o, h, c = dec(dec_in.unsqueeze(1), h, c)
            top = o.argmax(1).item()
            if top == tgt_v.w2i[EOS] or top == tgt_v.w2i[PAD]: break
            result.append(top)
            dec_in = torch.tensor([top]).to(device)
        translated = tgt_v.decode(result)
        b = calculate_bleu(tgt_text, translated)
        total_bleu += b
        print(f"{src_text} -> {translated} (BLEU: {b:.2f})")

avg_bleu = total_bleu / len(test_samples)
print(f"\n=== BLEU Score: {avg_bleu:.2f} ===")

# FASE 6: Deployment

In [ ]:
# ========================================
# CELDA 8: Guardar modelo (para reuse)
# ========================================

torch.save({
    'model_state_dict': model.state_dict(),
    'src_vocab': src_v.w2i,
    'tgt_vocab': tgt_v.w2i,
    'src_idx2word': src_v.i2w,
    'tgt_idx2word': tgt_v.i2w,
    'max_len': MAX_LEN,
    'bleu_score': avg_bleu,
}, 'translator_model.pt')

params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo guardado: translator_model.pt ({params:,} parámetros)")

In [ ]:
# ========================================
# CELDA 9: Función de traducción
# ========================================

def translate(text, direction="EN->ES"):
    if not text.strip(): return ""
    model.eval()
    with torch.no_grad():
        if direction == "ES->EN":
            src_enc = torch.tensor([tgt_v.encode(text, MAX_LEN)]).to(device)
            _, h, c = enc(src_enc)
            dec_in = torch.tensor([src_v.w2i[SOS]]).to(device)
            vocab = src_v
        else:
            src_enc = torch.tensor([src_v.encode(text, MAX_LEN)]).to(device)
            _, h, c = enc(src_enc)
            dec_in = torch.tensor([tgt_v.w2i[SOS]]).to(device)
            vocab = tgt_v
        result = []
        for _ in range(MAX_LEN):
            o, h, c = dec(dec_in.unsqueeze(1), h, c)
            top = o.argmax(1).item()
            if top == vocab.w2i[EOS] or top == vocab.w2i[PAD]: break
            result.append(top)
            dec_in = torch.tensor([top]).to(device)
        return vocab.decode(result)

# Prueba rápida
print("Prueba: hello ->", translate("hello", "EN->ES"))
print("Prueba: hola ->", translate("hola", "ES->EN"))

## 6.1 Interfaz Gradio (Listo para produção)

In [ ]:
# ========================================
# CELDA 10: Interfaz Gradio
# ========================================

with gr.Blocks() as demo:
    gr.Markdown("# Traductor RNN - UAC\n## Seq2Seq LSTM - CRISP-ML(Q)")
    gr.Markdown(f"**BLEU Score: {avg_bleu:.2f}** | **Parámetros: {params:,}**")
    with gr.Row():
        inp = gr.Textbox(label="Texto")
        direction = gr.Radio(["EN->ES", "ES->EN"], label="Dirección", value="EN->ES")
    btn = gr.Button("Traducir")
    out = gr.Textbox(label="Traducción")
    btn.click(fn=translate, inputs=[inp, direction], outputs=out)

demo.launch()

## 6.2 API con FastAPI

In [ ]:
# ========================================
# CELDA 11: FastAPI (copie a archivo separado)
# ========================================

print("""
# FastAPI - guardar como api.py y ejecutar: uvicorn api:app --reload
"""
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import torch.nn as nn
import numpy as np

app = FastAPI(title="Traductor RNN UAC")

class TranslateRequest(BaseModel):
    text: str
    direction: str = "EN->ES"

class TranslateResponse(BaseModel):
    translation: str
    direction: str
    bleu_score: float

@app.post("/translate", response_model=TranslateResponse)
def translate(req: TranslateRequest):
    result = translate(req.text, req.direction)  # usa función definida arriba
    return TranslateResponse(translation=result, direction=req.direction, bleu_score=avg_bleu)

@app.get("/health")
def health():
    return {"status": "ok", "bleu": avg_bleu, "params": params}

""" )

# FASE 6 (Q): Monitoring & Maintenance
## Sistema de Feedback

In [ ]:
# ========================================
# CELDA 12: Sistema de Feedback (Monitoring)
# ========================================

feedback_data = []

def submit_feedback(original, translation, correct_translation, direction):
    feedback_data.append({
        'original': original,
        'got': translation,
        'expected': correct_translation,
        'direction': direction
    })
    return f"Gracias! Feedback guardado: {len(feedback_data)} correcciones"

In [ ]:
# ========================================
# CELDA 13: Interfaz de Feedback
# ========================================

with gr.Blocks() as feedback_demo:
    gr.Markdown("# Sistema de Feedback - CRISP-ML(Q)\n## Ayude a mejorar el traductor")
    gr.Markdown("如果 el modelo cometió un error, sugiera la traducción correcta.")
    with gr.Row():
        fb_original = gr.Textbox(label="Texto original")
        fb_got = gr.Textbox(label="Traducción obtenida")
    with gr.Row():
        fb_correct = gr.Textbox(label="Traducción correcta")
        fb_direction = gr.Radio(["EN->ES", "ES->EN"], label="Dirección")
    fb_btn = gr.Button("Enviar Feedback")
    fb_output = gr.Textbox(label="Estado")
    fb_btn.click(fn=submit_feedback, inputs=[fb_original, fb_got, fb_correct, fb_direction], outputs=fb_output)
    gr.Markdown(f"### Feedback recibido: {len(feedback_data)}")

feedback_demo.launch()

## Resumen CRISP-ML(Q)
- Fase 1: Business Understanding ✓
- Fase 2: Data Preparation ✓
- Fase 3: Modeling (Seq2Seq LSTM) ✓
- Fase 4: Training ✓
- Fase 5: Evaluation (BLEU) ✓
- Fase 6: Deployment (Gradio + FastAPI) ✓
- Fase 6(Q): Monitoring (Feedback Loop) ✓